# Lecture 8 · Fine-tuning language models (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — Load the questions and set the evaluation sets aside

[1. Decide whether the model needs training](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/01-decide-whether-the-model-needs-training/)

In [ ]:
import json, random, urllib.request, pathlib
from collections import Counter

DATA_URL = ("https://aiml-study-guides.onrender.com/datasets/generative-"
            "agentic-ai/l8-questions.jsonl")
path = pathlib.Path("l8-questions.jsonl")
if not path.exists():
    urllib.request.urlretrieve(DATA_URL, path)
rows = [json.loads(line) for line in path.open(encoding="utf-8")]
print(len(rows), "questions; per lecture:",
      dict(sorted(Counter(r["lecture"] for r in rows).items())),
      "(0 = not in the notes)")

# Questions written from the same section are near-paraphrases, so a section is
# one "family":
# all of its questions go to the same side of the split. Each off-topic
# question is its own family.
for i, r in enumerate(rows):
    r["family"] = r["section"] if r["lecture"] else f"none-{i}"
random.seed(0)
families = sorted({r["family"] for r in rows})
random.shuffle(families)
fifth = len(families) // 5
test_families, val_families = set(families[:fifth]), set(families[fifth:2 *
                                                                  fifth])
# looked at once, at the very end (Step 16)
test = [r for r in rows if r["family"] in test_families]
# every choice is made on this one
val = [r for r in rows if r["family"] in val_families]
train = [r for r in rows if r["family"] not in test_families | val_families]
print(
    f"{len(train)} train, {len(val)} validation, {len(test)} test; sections "
    f"shared between splits:",
    len({r["family"]
         for r in train} & {r["family"]
                            for r in val + test}))
for r in val[:3]:
    print(f'  {r["lecture"]} | {r["question"]}')

## Step 2 — Ask the small model with a prompt

[1. Decide whether the model needs training](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/01-decide-whether-the-model-needs-training/)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"  # 135 M parameters, about 270 MB
tok = AutoTokenizer.from_pretrained(BASE)
# bf16 on disk; most laptop CPUs compute bf16 slowly
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)

NAMES = {
    1: "Lecture 1 · Generative and agentic AI",
    2: "Lecture 2 · Training language models",
    3: "Lecture 3 · Choosing and configuring models",
    4: "Lecture 4 · Context engineering", 5: "Lecture 5 · LangChain",
    6: "Lecture 6 · Agents and tools", 0: "Not in the notes"
}
ANSWERS = [NAMES[n] for n in range(7)]
LONG_PROMPT = (
    "Which lecture of the course notes answers the question? Reply "
    "with one line: " + "; ".join(ANSWERS[1:]) + "; or Not in the notes.")

def prompt_ids(question, system):
    """Render a question under a system prompt; return its token ids."""
    text = tok.apply_chat_template([{"role": "system", "content": system},
                                    {"role": "user", "content": question}],
                                   add_generation_prompt=True, tokenize=False)
    return tok(text, add_special_tokens=False)["input_ids"]

ANSWER_IDS = [
    tok(a + tok.eos_token, add_special_tokens=False)["input_ids"]
    for a in ANSWERS
]

@torch.no_grad()
def route(question, system, net=None, per_token=False):
    """Score each of the 7 allowed answers by its log-probability (or its
    mean per token); return the best."""
    net = net or model
    p = prompt_ids(question, system)
    seqs = [p + a for a in ANSWER_IDS]
    L = max(map(len, seqs))
    ids = torch.tensor([s + [tok.pad_token_id] * (L - len(s)) for s in seqs])
    mask = (torch.arange(L)[None]
            < torch.tensor([len(s) for s in seqs])[:, None]).long()
    logp = net(input_ids=ids, attention_mask=mask).logits.log_softmax(-1)
    scores = [
        sum(logp[j, len(p) - 1 + t, a[t]].item()
            for t in range(len(a))) / (len(a) if per_token else 1)
        for j, a in enumerate(ANSWER_IDS)
    ]
    return max(range(7), key=scores.__getitem__)

def accuracy(questions, system, net=None, per_token=False):
    """Route each question; return (share routed right, predictions)."""
    preds = [route(r["question"], system, net, per_token) for r in questions]
    return sum(p == r["lecture"]
               for p, r in zip(preds, questions)) / len(questions), preds

ids = torch.tensor([prompt_ids(val[0]["question"], LONG_PROMPT)])
reply = model.generate(ids, attention_mask=torch.ones_like(ids),
                       max_new_tokens=25, do_sample=False)
print("free reply:",
      repr(tok.decode(reply[0, ids.shape[1]:], skip_special_tokens=True)))
acc, preds = accuracy(val, LONG_PROMPT)
print(f"prompt only, best of the 7 allowed answers: "
      f"validation accuracy {acc:.2f}")
print("most chosen:", Counter(ANSWERS[p] for p in preds).most_common(2))
# shorter answers no longer win by length
acc_mean, preds_mean = accuracy(val, LONG_PROMPT, per_token=True)
print(
    f"same, mean log-probability per token: validation accuracy "
    f"{acc_mean:.2f}; most chosen:",
    Counter(ANSWERS[p] for p in preds_mean).most_common(2))

## Step 3 — Try a classic classifier

[1. Decide whether the model needs training](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/01-decide-whether-the-model-needs-training/)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# words and word pairs, damped counts
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
clf = LogisticRegression(max_iter=3000, C=10)
clf.fit(vec.fit_transform([r["question"] for r in train]),
        [r["lecture"] for r in train])
tfidf_pred = list(clf.predict(vec.transform([r["question"] for r in val])))
tfidf_acc = sum(p == r["lecture"] for p, r in zip(tfidf_pred, val)) / len(val)
print(f"TF-IDF + logistic regression: validation accuracy {tfidf_acc:.2f}  "
      f"({vec.transform(['x']).shape[1]:,} features, trained in well "
      f"under a second)")

## Step 4 — Ask a large hosted model 💳 API

[1. Decide whether the model needs training](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/01-decide-whether-the-model-needs-training/)

In [ ]:
import os, re, time
from openai import OpenAI

# Same three variables as Lectures 6 and 7.
LLM_READY = bool(os.environ.get("LLM_API_KEY"))
client = OpenAI(api_key=os.environ["LLM_API_KEY"],
                base_url=os.environ.get("LLM_BASE_URL")) if LLM_READY else None
MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")

def parse(reply):
    """Map a reply to its lecture (1–6), 0 if off-topic, None if broken."""
    m = re.search(r"Lecture\s*([1-6])", reply)
    if m:
        return int(m.group(1))
    # None = the reply broke the format
    return 0 if "not in the notes" in reply.lower() else None

if LLM_READY:
    t0, tokens_in, tokens_out, api_pred = time.time(), 0, 0, []
    for r in val:
        out = client.chat.completions.create(
            model=MODEL, temperature=0, max_tokens=20,
            messages=[{"role": "system", "content": LONG_PROMPT},
                      {"role": "user", "content": r["question"]}])
        api_pred.append(parse(out.choices[0].message.content))
        tokens_in += out.usage.prompt_tokens
        tokens_out += out.usage.completion_tokens
    api_seconds = (time.time() - t0) / len(val)
    api_acc = sum(p == r["lecture"] for p, r in zip(api_pred, val)) / len(val)
    print(f"{MODEL}, prompt only: validation accuracy {api_acc:.2f}; format "
          f"errors {api_pred.count(None)}; "
          f"{api_seconds:.2f} s per question; tokens per question "
          f"{tokens_in / len(val):.0f} in, {tokens_out / len(val):.1f} out")

## Step 5 — Measure the loss of one target answer

[2. Decide what the training should teach](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/02-decide-what-the-training-should-teach/)

In [ ]:
SHORT_PROMPT = ("Route the question to the lecture of the course notes "
                "that answers it.")

r = train[0]
p, a = prompt_ids(r["question"], SHORT_PROMPT), ANSWER_IDS[r["lecture"]]
with torch.no_grad():
    logp = model(torch.tensor([p + a])).logits[0].log_softmax(-1)
print(r["question"], "→", ANSWERS[r["lecture"]])
nll = []
# the logits at position i predict the token at i + 1
for t, token in enumerate(a):
    nll.append(abs(logp[len(p) - 1 + t, token].item()))      # -log p ≥ 0
    print(f"  {tok.decode([token])!r:>14}   -log p = {nll[-1]:5.2f}")
print(f"SFT loss of this record = mean over {len(a)} answer tokens = "
      f"{sum(nll) / len(nll):.2f}")

## Step 6 — Write chat records and render one with the chat template

[3. Build data that teaches the intended behaviour](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/03-build-data-that-teaches-the-intended-behaviour/)

In [ ]:
def chat_record(r):
    """Take a question row; return its system/user/assistant chat record."""
    return {
        "messages": [{"role": "system", "content": SHORT_PROMPT},
                     {"role": "user", "content": r["question"]},
                     {"role": "assistant", "content": ANSWERS[r["lecture"]]}]
    }

# one JSON object per line: the usual SFT file
with open("train.jsonl", "w", encoding="utf-8") as f:
    for r in train:
        f.write(json.dumps(chat_record(r), ensure_ascii=False) + "\n")
print(open("train.jsonl", encoding="utf-8").readline().strip())
print(
    tok.apply_chat_template(chat_record(train[0])["messages"], tokenize=False))

## Step 7 — Check lengths, duplicates and questions that lean on an unseen passage

[3. Build data that teaches the intended behaviour](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/03-build-data-that-teaches-the-intended-behaviour/)

In [ ]:
import re

def words(q):
    """Take a question; return the set of its lower-case words and numbers."""
    return set(re.findall(r"[a-z0-9]+", q.lower()))

lengths = sorted(len(r["question"].split()) for r in rows)
print("words per question: min", lengths[0], "median",
      lengths[len(lengths) // 2], "max", lengths[-1])
print("exact duplicates:",
      len(rows) - len({r["question"].lower()
                       for r in rows}))
near = [(a["question"], b["question"]) for i, a in enumerate(rows)
        for b in rows[i + 1:]
        if len(words(a["question"]) & words(b["question"])) /
        len(words(a["question"]) | words(b["question"])) >= 0.8]
print("near duplicates (≥ 80 % shared words):", len(near))
for x, y in near[:2]:
    print("  ", x, "|", y)
VAGUE = re.compile(
    r"\b(this|the) (passage|text|section)\b|\bthe (example|code|figure)( "
    r"provided| shown| above)?\b|"
    r"\bin the (context|code) (provided|above|shown)\b|\bin this context\b",
    re.I)
vague = [r for r in rows if VAGUE.search(r["question"])]
print("questions that lean on an unseen passage:", len(vague))
for r in vague[:3]:
    print("  ", r["lecture"], "|", r["question"])

## Step 8 — Compare a random split with a split by section

[3. Build data that teaches the intended behaviour](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/03-build-data-that-teaches-the-intended-behaviour/)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

def nn_accuracy(tr, dv):
    """1-nearest-neighbour on TF-IDF: copies the label of the most similar
    training question."""
    v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    knn = KNeighborsClassifier(1, metric="cosine").fit(
        v.fit_transform([r["question"] for r in tr]),
        [r["lecture"] for r in tr])
    return sum(
        p == r["lecture"]
        for p, r in zip(knn.predict(v.transform([r["question"]
                                                 for r in dv])), dv)) / len(dv)

for seed in range(5):
    rng = random.Random(seed)
    fams = sorted({r["family"] for r in rows}); rng.shuffle(fams)
    held = set(fams[: len(fams) // 5])
    fam_dev = [r for r in rows if r["family"] in held]
    shuffled = rows[:]; rng.shuffle(shuffled)
    leaky_dev, leaky_train = shuffled[: len(fam_dev)], shuffled[len(fam_dev):]
    print(
        f"seed {seed}: split by family "
        + format(nn_accuracy([r for r in rows if r['family'] not in held],
                             fam_dev), ".2f")
        + f"   random split {nn_accuracy(leaky_train, leaky_dev):.2f}")

## Step 9 — Count trainable parameters and memory for full fine-tuning, LoRA and QLoRA

[4. Choose which parameters to train](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/04-choose-which-parameters-to-train/)

In [ ]:
def lora_count(h, inter, kv, n_layers, r):
    """Trainable LoRA parameters on all 7 projections: r × (d_in + d_out)
    per matrix (A is r × d_in, B is d_out × r)."""
    shapes = [(h, h), (h, kv), (h, kv), (h, h), (h, inter), (h, inter),
              (inter, h)]
    return n_layers * sum(r * (i + o) for i, o in shapes)

cfg = model.config
kv = cfg.hidden_size // cfg.num_attention_heads * cfg.num_key_value_heads
total = sum(p.numel() for p in model.parameters())
for r in (4, 16, 64):
    n = lora_count(cfg.hidden_size, cfg.intermediate_size, kv,
                   cfg.num_hidden_layers, r)
    print(f"SmolLM2-135M, LoRA r={r:<3}: {n:>10,} trainable = "
          f"{n / total:.2%} of {total:,}")

def memory_gb(n_params, n_lora, method):
    """Weights + gradients + optimizer state in GB; activations are extra."""
    # bf16 weights 2 + grads 2 + fp32 master 4 + Adam 8 bytes
    if method == "full":  return n_params * 16 / 1e9
    # frozen bf16 weights; adapters trained like above
    if method == "lora":  return (n_params * 2 + n_lora * 16) / 1e9
    # NF4: 4 bits + quantization constants
    if method == "qlora": return (n_params * 0.516 + n_lora * 16) / 1e9

# (parameters, hidden size, MLP size, key/value width, layers): SmolLM2-135M,
# Llama 2 7B, Llama 2 70B
for name, n, h, inter, kvw, layers in [
    ("SmolLM2-135M", total, cfg.hidden_size, cfg.intermediate_size, kv,
     cfg.num_hidden_layers), ("Llama-2-7B", 6.74e9, 4096, 11008, 4096, 32),
    ("Llama-2-70B", 68.98e9, 8192, 28672, 1024, 80)
]:
    nl = lora_count(h, inter, kvw, layers, 16)
    print(f"{name:>12}: full {memory_gb(n, nl, 'full'):7.1f} GB | LoRA r=16 "
          f"{memory_gb(n, nl, 'lora'):6.1f} GB"
          f" | QLoRA r=16 {memory_gb(n, nl, 'qlora'):6.1f} GB | adapter "
          f"{nl / 1e6:5.1f} M params")

## Step 10 — Mark which tokens of a record count toward the loss

[5. Run a controlled training experiment](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/05-run-a-controlled-training-experiment/)

In [ ]:
def training_record(r):
    """Token ids of prompt + answer, and labels: -100 (ignored by the loss)
    on every prompt token."""
    p, a = prompt_ids(r["question"], SHORT_PROMPT), ANSWER_IDS[r["lecture"]]
    return p + a, [-100] * len(p) + a

ids, labels = training_record(train[0])
shown = [
    f"[{tok.decode([i])}]" if l != -100 else tok.decode([i])
    for i, l in zip(ids, labels)
]
print("".join(shown).replace("\n", "⏎"))
print(f"{len(ids)} tokens, {sum(l != -100 for l in labels)} of them in the "
      f"loss (the [bracketed] ones)")

## Step 11 — Sanity run: memorise 16 training examples

[5. Run a controlled training experiment](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/05-run-a-controlled-training-experiment/)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import get_cosine_schedule_with_warmup

def batches(records, size):
    """Pad a list of (ids, labels) to one length; the attention mask hides
    the padding."""
    L = max(len(i) for i, _ in records)
    ids = torch.tensor(
        [i + [tok.pad_token_id] * (L - len(i)) for i, _ in records])
    lab = torch.tensor([l + [-100] * (L - len(l)) for _, l in records])
    mask = (torch.arange(L)[None]
            < torch.tensor([len(i) for i, _ in records])[:, None]).long()
    return ids, mask, lab

TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
]

def new_lora(rank):
    """Take a rank; return the base model with a fresh LoRA adapter on it."""
    torch.manual_seed(0)
    base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)
    return get_peft_model(
        base,
        LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.05,
                   target_modules=TARGETS, task_type="CAUSAL_LM"))

# Sanity run: can the setup memorise 16 examples? If not, something is wrong
# before any tuning matters.
tiny = train[::37][:16]
probe = new_lora(16)
opt = torch.optim.AdamW([p for p in probe.parameters() if p.requires_grad],
                        lr=3e-3)
ids, mask, lab = batches([training_record(r) for r in tiny], 16)
for step in range(1, 61):
    loss = probe(input_ids=ids, attention_mask=mask, labels=lab).loss
    loss.backward(); opt.step(); opt.zero_grad()
    if step % 15 == 0:
        print(f"step {step:>2}: loss {loss.item():.3f}")
probe.eval()  # dropout off before measuring
print(f"memorised: {accuracy(tiny, SHORT_PROMPT, probe)[0]:.2f} of the 16 "
      f"training examples")

## Step 12 — Train the LoRA adapter and log training and validation loss

[5. Run a controlled training experiment](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/05-run-a-controlled-training-experiment/)

In [ ]:
import os, time

# settings: Section 5.3 and Step 13 show how they were chosen
RANK, LR, STEPS, BATCH = 4, 1e-3, 600, 8
lora = new_lora(RANK)
lora.print_trainable_parameters()
opt = torch.optim.AdamW([p for p in lora.parameters() if p.requires_grad],
                        lr=LR)
# 5 % warm-up, then a cosine decay
schedule = get_cosine_schedule_with_warmup(opt, STEPS // 20, STEPS)
train_records = [training_record(r) for r in train]
val_records = [training_record(r) for r in val]

@torch.no_grad()
def val_loss(net):
    """Take a model; return its mean loss per scored validation token."""
    net.eval(); total = count = 0
    for k in range(0, len(val_records), 16):
        ids, mask, lab = batches(val_records[k:k + 16], 16)
        n = (lab[:, 1:] != -100).sum().item()
        total += net(input_ids=ids, attention_mask=mask,
                     labels=lab).loss.item() * n
        count += n
    net.train(); return total / count

random.seed(0); t0, window = time.time(), []
for step in range(1, STEPS + 1):
    ids, mask, lab = batches(random.sample(train_records, BATCH), BATCH)
    loss = lora(input_ids=ids, attention_mask=mask, labels=lab).loss
    loss.backward(); opt.step(); schedule.step(); opt.zero_grad()
    window.append(loss.item())
    if step % 100 == 0:
        print(f"step {step:>3}: train loss {sum(window) / len(window):.3f}   "
              f"validation loss {val_loss(lora):.3f}   "
              f"{time.time() - t0:4.0f} s")
        window = []
lora.eval()  # dropout off: from here on the adapter is only used, not trained
lora_acc, lora_pred = accuracy(val, SHORT_PROMPT, lora)
print(f"LoRA r={RANK}: validation accuracy {lora_acc:.2f};  on {len(val)} "
      f"training questions "
      f"{accuracy(train[:len(val)], SHORT_PROMPT, lora)[0]:.2f}")

## Step 13 — Read six recorded runs that change one setting at a time

[5. Run a controlled training experiment](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/05-run-a-controlled-training-experiment/)

In [ ]:
GRID_URL = ("https://aiml-study-guides.onrender.com/datasets/generative-"
            "agentic-ai/l8-lora-grid.json")
grid_path = pathlib.Path("l8-lora-grid.json")
if not grid_path.exists():
    urllib.request.urlretrieve(GRID_URL, grid_path)
# recorded runs, 600 steps each: one setting changed at a time
grid = json.loads(grid_path.read_text())

ranks, rates = sorted({g["rank"]
                       for g in grid}), sorted({g["lr"]
                                                for g in grid})
print("validation accuracy after 600 steps (training accuracy in brackets)")
print("rank \\ lr " + "".join(f"{lr:>14g}" for lr in rates))
for r in ranks:
    cells = {g["lr"]: g for g in grid if g["rank"] == r}
    print(f"{r:>9} " + "".join(
        f"{cells[lr]['dev_acc']:>8.2f} ({cells[lr]['train_acc']:.2f})" if lr in
        cells else f"{'—':>14}" for lr in rates))
best = max(grid, key=lambda g: g["dev_acc"])
low = min(best["curve"][1:], key=lambda c: c["val_loss"])
print(f"best: rank {best['rank']}, lr {best['lr']:g}; its validation loss was "
      f"lowest at step {low['step']} ({low['val_loss']:.3f}),"
      f" {best['curve'][-1]['val_loss']:.3f} at the end")

## Step 14 — Precision, recall and the confusion matrix on validation

[6. Establish whether the adaptation helped](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/06-establish-whether-the-adaptation-helped/)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

truth = [r["lecture"] for r in val]
short = ["none", "L1", "L2", "L3", "L4", "L5", "L6"]
print(
    classification_report(truth, lora_pred, labels=range(7),
                          target_names=short, digits=2, zero_division=0))
print("rows = true, columns = predicted")
print("      " + " ".join(f"{s:>4}" for s in short))
for name, line in zip(short, confusion_matrix(truth, lora_pred,
                                              labels=range(7))):
    print(f"{name:>5} " + " ".join(f"{v:>4}" for v in line))

## Step 15 — Ask general questions with the adapter off and on

[6. Establish whether the adaptation helped](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/06-establish-whether-the-adaptation-helped/)

In [ ]:
GENERAL = [
    "Write one sentence about autumn.", "What is 17 + 25?",
    "Translate 'good morning' into French."
]

def reply(net, text, system="You are a helpful assistant."):
    """Take a model, request and system text; return its 20-token reply."""
    ids = torch.tensor([prompt_ids(text, system)])
    out = net.generate(ids, attention_mask=torch.ones_like(ids),
                       max_new_tokens=20, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:],
                      skip_special_tokens=True).strip().replace("\n", " ")

for text in GENERAL:
    # the same weights without the adapter = the base model
    with lora.disable_adapter():
        before = reply(lora, text)
    print(f"{text}\n  base:    {before}\n  adapter: {reply(lora, text)}")

## Step 16 — Every method once on the test set

[6. Establish whether the adaptation helped](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/06-establish-whether-the-adaptation-helped/)

In [ ]:
# The test set, used once at the end: every method on the same 148 questions,
# with the settings fixed beforehand.
def score(pred):
    """Take predicted labels for test; return the share that are correct."""
    return sum(p == r["lecture"] for p, r in zip(pred, test)) / len(test)

t0 = time.time()
lora_test = [route(r["question"], SHORT_PROMPT, lora) for r in test]
lora_s = (time.time() - t0) / len(test)
t0 = time.time()
tfidf_test = list(clf.predict(vec.transform([r["question"] for r in test])))
tfidf_s = (time.time() - t0) / len(test)
small_test = [route(r["question"], LONG_PROMPT, per_token=True) for r in test]
print(f"{'method':<40}{'test accuracy':>14}{'s / question':>14}")
print(
    f"{'SmolLM2-135M, prompt only (per token)':<40}{score(small_test):>14.2f}")
print(
    f"{'TF-IDF + logistic regression':<40}"
    f"{score(tfidf_test):>14.2f}{tfidf_s:>14.4f}"
)
print(
    f"{'SmolLM2-135M + LoRA (this CPU)':<40}"
    f"{score(lora_test):>14.2f}{lora_s:>14.3f}"
)
if LLM_READY:
    t0 = time.time(); api_test = []
    for r in test:
        out = client.chat.completions.create(
            model=MODEL, temperature=0, max_tokens=20,
            messages=[{"role": "system", "content": LONG_PROMPT},
                      {"role": "user", "content": r["question"]}])
        api_test.append(parse(out.choices[0].message.content))
    print(
        f"{MODEL + ', prompt only (network)':<40}{score(api_test):>14.2f}"
        f"{(time.time() - t0) / len(test):>14.3f}"
    )

# Where do the two methods differ? Question by question, and per label for the
# router.
L = [p == r["lecture"] for p, r in zip(lora_test, test)]
T = [p == r["lecture"] for p, r in zip(tfidf_test, test)]
print(f"\ncorrect of {len(test)}: LoRA {sum(L)}, TF-IDF {sum(T)}; right only "
      f"with LoRA {sum(l and not t for l, t in zip(L, T))},"
      f" only with TF-IDF {sum(t and not l for l, t in zip(L, T))}")
print(
    "LoRA per label on test (correct/questions):",
    ", ".join(f"{ANSWERS[k].split(' · ')[0]} "
              f"{sum(l for l, r in zip(L, test) if r['lecture'] == k)}/"
              f"{sum(r['lecture'] == k for r in test)}" for k in range(7)))

## Step 17 — Save the adapter, merge it and check that the merged model routes the same

[7. Deploy the right artifacts and keep checking them](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-08-fable/07-deploy-the-right-artifacts-and-keep-checking-them/)

In [ ]:
import copy, pathlib
from peft import PeftModel

# writes only the adapter: config + trained matrices
lora.save_pretrained("router-adapter")
size = sum(f.stat().st_size for f in pathlib.Path("router-adapter").iterdir())
print(f"adapter on disk: {size / 1e6:.1f} MB (the base model it needs: about "
      f"540 MB in float32, 270 MB in bf16)")

reloaded = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32),
    "router-adapter")
# W ← W + (α/r)·B·A, then the adapter layers are gone
merged = reloaded.merge_and_unload()
same = sum(
    route(r["question"], SHORT_PROMPT, merged) == p
    for r, p in zip(val, lora_pred))
print(f"merged model agrees with the adapter on {same} of {len(val)} "
      f"validation questions")
print(
    f"merged: {sum(p.numel() for p in merged.parameters()):,} parameters, no "
    f"adapter layers left: "
    f"{not any('lora' in n for n, _ in merged.named_parameters())}")

# Shipping the merged file in bf16 halves it: every weight is rounded to bf16
# (computed in float32 here).
shipped = copy.deepcopy(merged)
for p in shipped.parameters():
    p.data = p.data.to(torch.bfloat16).float()
same16 = sum(
    route(r["question"], SHORT_PROMPT, shipped) == p
    for r, p in zip(val, lora_pred))
print(f"rounded to bf16 for shipping: agrees with the adapter on "
      f"{same16} of {len(val)}")
w = "model.layers.0.self_attn.q_proj.weight"
W0 = model.state_dict()[w]  # Step 2's base, untouched
update = merged.state_dict()[w] - W0                  # what the merge added
kept = shipped.state_dict()[w] - W0.to(torch.bfloat16).float()
print(f"{w}: bf16 rounding changed that update by "
      f"{(kept - update).norm() / update.norm():.0%} of its size")